# Manovia ML — first real run: GoEmotions (English)

Ready-to-run Colab/Kaggle notebook for the Day-31 platform. It does **not** reimplement training:
it installs this repository and calls the same modules the sandbox smoke run calls
(`manovia_ml.training.train`, `manovia_ml.evaluation.evaluate`). If the numbers here differ from a
smoke run, they differ because the data differs — which is the point.

**Read first:** `ml/data/CATALOGUE.md` § 1 (GoEmotions, Apache-2.0, *use with conditions*: no
redistribution of rows, treat them as personal data) and § "Hindi and Bengali emotion data"
for what this run cannot tell us about Hindi/Bengali.

Runtime: CPU is enough (this is a TF-IDF + linear model). A GPU runtime will not make it faster.
Expected wall clock: ~5 min including the download.

**Never commit the corpus or the model weights.** What you bring back to the repository: the
report (`ml/reports/`), the recorded baseline (`ml/configs/expected/`), and the run's
`environment.json` / `data.json` hashes.

In [ ]:
# 1. Get the repository and install the pinned environments.
!git clone -q https://github.com/BuildbyArindam/Manovia.git /content/Manovia
%cd /content/Manovia
!pip install -q -e "./backend[dev]"
!pip install -q -e "./ml[ml,dev]"
!pip install -q datasets

In [ ]:
# 2. Regenerate the committed toy splits (offline) so the smoke run at the end works.
%cd /content/Manovia/ml
!python -m dvc repro
!python -m dvc status

In [ ]:
# 3. Download GoEmotions and convert it to Manovia's label scheme.
#    The conversion is pure and tested offline (ml/tests/test_goemotions.py);
#    only this download touches the network.
for split in ("train", "validation", "test"):
    !python -m manovia_ml.data.goemotions --download {split} --split-name {split} \
        --out data/raw/goemotions/{split}.jsonl

In [ ]:
# 4. Look at what the conversion did before training on it: label counts, how many rows were
#    dropped because no GoEmotions class maps onto a Manovia emotion, and the unmapped pile.
import json

for split in ("train", "validation", "test"):
    stats = json.load(open(f"data/raw/goemotions/{split}.stats.json"))
    print(split, stats["rows"], "rows |", stats["by_label"])

# Known gaps to carry into the model card:
#   * no `loneliness` class exists in GoEmotions at all;
#   * multi-label rows collapse to one label by priority (PRIORITY in data/goemotions.py);
#   * Reddit register is a domain shift from a wellbeing companion.

In [ ]:
# 5. Version the bytes locally (Colab VM only) and print the hashes that will go into the run.
!python -m dvc add data/raw/goemotions
!sha256sum data/raw/goemotions/*.jsonl

In [ ]:
# 6. Train. MLflow writes to ml/mlruns (local file store) on this VM.
#    The run prints RUN_ID plus the metrics; the same config + seed must reproduce them.
!python -m manovia_ml.training.train --config configs/colab_goemotions_logreg.yaml --json

In [ ]:
# 7. Re-score the run from its own artefacts and check the two agree.
run_id = input("RUN_ID from step 6: ").strip()
!python -m manovia_ml.evaluation.evaluate --run {run_id}

In [ ]:
# 8. Look at the confusion matrix line by line before believing macro-F1.
import json
from importlib import import_module

artifacts = import_module("mlflow").artifacts
path = artifacts.download_artifacts(run_id=run_id, artifact_path="evaluation.json")
evaluation = json.load(open(path))
labels = evaluation["labels"]
print("        " + " ".join(f"{label[:7]:>7}" for label in labels))
for label, row in zip(labels, evaluation["confusion_matrix"]["rows"]):
    print(f"{label[:7]:>7} " + " ".join(f"{value:>7}" for value in row))
print()
print(evaluation["per_class_f1"])

In [ ]:
# 9. Write the report to bring back into the repository (small, no data, no weights).
!python -m manovia_ml.reporting --run {run_id} --config-path configs/colab_goemotions_logreg.yaml

# Bring back exactly these files:
#   ml/reports/<run_id>.md            (the report)
#   ml/configs/expected/colab_goemotions_logreg.json   (record the baseline deliberately)
#   the run's environment.json/data.json if you want the hashes quoted somewhere
# Do NOT bring back: data/raw/**, ml/mlruns/**, any *.joblib.

In [ ]:
# 10. Before you leave: the smoke run must still pass on the same platform.
!python -m manovia_ml.training.train --config configs/toy_emotion_sgd.yaml --phase smoke